# FastAI LSTM Sequential Finetuning Tutorial

This notebook demonstrates **sequential finetuning** of an LSTM model using **FastAI primitives**.

## Pipeline
1. **Stage 1**: Train on UTM data (universal computation)
2. **Stage 2**: Fine-tune on CTW data (sequence compression)

**Why LSTM?** Better length generalization than Transformers ([Deletang et al., 2024](https://arxiv.org/html/2401.14953v1))

**Why FastAI?** One-cycle LR, automatic metrics, less code

In [ ]:
# Imports
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
from fastai.learner import Learner
from fastai.optimizer import Adam

from data import utm_data_generator as utm_dg, ctw_data_generator as ctw_dg, utms as utms_lib
from torch_models.lstm import LSTMConfig, LSTMDecoderLM
from torch_training.fastai_dataloaders import create_utm_dataloaders, create_ctw_dataloaders
from torch_training.fastai_callbacks import PerplexityMetric, GradNormCallback, LSTMLogProbLoss

print(f"✓ Imports successful | PyTorch: {torch.__version__} | Device: {torch.cuda.is_available() and 'cuda' or 'cpu'}")

In [ ]:
# Configuration
device = "cuda" if torch.cuda.is_available() else "cpu"
VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS = 128, 128, 256, 2
BATCH_SIZE, SEQ_LENGTH = 32, 256
UTM_EPOCHS, CTW_EPOCHS = 5, 3
MAX_LR, BATCHES_PER_EPOCH = 1e-3, 50
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Config: {NUM_LAYERS}-layer LSTM, {HIDDEN_DIM} hidden, {UTM_EPOCHS}+{CTW_EPOCHS} epochs, lr={MAX_LR}")

In [ ]:
# Create LSTM Model
config = LSTMConfig(vocab_size=VOCAB_SIZE, embedding_dim=EMBEDDING_DIM, 
                    hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, dropout=0.2, tie_weights=True)
model = LSTMDecoderLM(config).to(device)
num_params = sum(p.numel() for p in model.parameters())
print(f"✓ Model created: {num_params:,} parameters")

# Test forward pass
with torch.no_grad():
    test_out = model(torch.randint(0, VOCAB_SIZE, (2, 16)).to(device))
print(f"✓ Forward pass: {test_out.shape}, log_probs: {(test_out <= 0).all()}")

In [ ]:
# Create UTM Data Generator
rng_utm = np.random.default_rng(seed=SEED)
program_sampler = utms_lib.FastSampler(rng=rng_utm)
utm = utms_lib.BrainPhoqueUTM(program_sampler)

utm_generator = utm_dg.UTMDataGenerator(
    batch_size=BATCH_SIZE, seq_length=SEQ_LENGTH, rng=rng_utm, utm=utm,
    memory_size=10, maximum_steps=100, tokenizer=utm_dg.Tokenizer.ASCII, maximum_program_length=100
)

print(f"✓ UTM generator: batch={BATCH_SIZE}, seq={SEQ_LENGTH}, mem=10, steps=100")

In [ ]:
# Create FastAI DataLoaders for UTM
dls_utm = create_utm_dataloaders(utm_generator, batches_per_epoch=BATCHES_PER_EPOCH, device=device)
print(f"✓ UTM DataLoaders: {BATCHES_PER_EPOCH} train batches, {BATCHES_PER_EPOCH//10} valid batches")

In [ ]:
# Create FastAI Learner
learn_utm = Learner(dls=dls_utm, model=model, loss_func=LSTMLogProbLoss(),
                    opt_func=Adam, metrics=[PerplexityMetric()])
print("✓ Learner created with Adam optimizer and Perplexity metric")

In [ ]:
# STAGE 1: Train on UTM Data
print("="*60)
print(f"STAGE 1: Training on UTM Data ({UTM_EPOCHS} epochs)")
print("="*60)

learn_utm.fit_one_cycle(n_epoch=UTM_EPOCHS, lr_max=MAX_LR, cbs=[GradNormCallback()])
print("\n✓ Stage 1 complete!")

In [ ]:
# Analyze Stage 1
print(f"UTM Training Results:")
print(f"  Final train loss: {learn_utm.recorder.values[-1][0]:.4f}")
print(f"  Final valid loss: {learn_utm.recorder.values[-1][1]:.4f}")
print(f"  Final perplexity: {learn_utm.recorder.values[-1][2]:.2f}")

try:
    learn_utm.recorder.plot_loss()
except: pass

In [ ]:
# Test Generation after UTM training
model.eval()
with torch.no_grad():
    prompt = torch.randint(0, VOCAB_SIZE, (1, 10)).to(device)
    generated = model.generate(prompt=prompt, max_length=50, temperature=1.0)

print(f"Generated sequence (after UTM): length={generated.shape[1]}")
try:
    chars = ''.join([chr(t) if 32 <= t < 127 else '?' for t in generated[0, :50].tolist()])
    print(f"  ASCII: {chars}")
except: pass

In [ ]:
# Create CTW Data Generator
rng_ctw = np.random.default_rng(seed=SEED + 1)
ctw_generator = ctw_dg.CTWGenerator(
    batch_size=BATCH_SIZE, seq_length=SEQ_LENGTH, rng=rng_ctw, max_depth=5, with_contexts=False
)
print(f"✓ CTW generator: batch={BATCH_SIZE}, seq={SEQ_LENGTH}, depth=5")

In [ ]:
# Create FastAI DataLoaders for CTW
dls_ctw = create_ctw_dataloaders(ctw_generator, batches_per_epoch=BATCHES_PER_EPOCH, device=device)
learn_utm.dls = dls_ctw
print(f"✓ CTW DataLoaders created and Learner updated")

In [ ]:
# STAGE 2: Fine-tune on CTW Data
print("="*60)
print(f"STAGE 2: Fine-tuning on CTW Data ({CTW_EPOCHS} epochs)")
print("="*60)

learn_utm.fit_one_cycle(n_epoch=CTW_EPOCHS, lr_max=MAX_LR*0.5, cbs=[GradNormCallback()])
print("\n✓ Stage 2 complete!")

In [ ]:
# Analyze Stage 2
print(f"CTW Fine-tuning Results:")
print(f"  Final train loss: {learn_utm.recorder.values[-1][0]:.4f}")
print(f"  Final valid loss: {learn_utm.recorder.values[-1][1]:.4f}")
print(f"  Final perplexity: {learn_utm.recorder.values[-1][2]:.2f}")

try:
    learn_utm.recorder.plot_loss()
except: pass

In [ ]:
# Save Final Model
from pathlib import Path
output_dir = Path("./checkpoints/notebook_fastai")
output_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = output_dir / "final_model.pth"
torch.save({"model_state_dict": model.state_dict(), "config": config.__dict__}, checkpoint_path)
print(f"✓ Model saved to: {checkpoint_path}")

In [ ]:
# Final Evaluation
learn_utm.dls = dls_utm
utm_loss = learn_utm.validate()[0]

learn_utm.dls = dls_ctw
ctw_loss = learn_utm.validate()[0]

print("\n" + "="*60)
print("FINAL EVALUATION")
print("="*60)
print(f"UTM Loss: {utm_loss:.4f}, Perplexity: {np.exp(utm_loss):.2f}")
print(f"CTW Loss: {ctw_loss:.4f}, Perplexity: {np.exp(ctw_loss):.2f}")
print("\n✓ Sequential finetuning complete!")

## Summary

**What we did:**
1. ✅ Created LSTM model with weight tying
2. ✅ Trained on UTM data (Stage 1)
3. ✅ Fine-tuned on CTW data (Stage 2)
4. ✅ Used FastAI's one-cycle policy
5. ✅ Tracked perplexity metrics
6. ✅ Saved final model

**Key FastAI Benefits:**
- Automatic LR scheduling
- Rich progress bars
- Easy metrics tracking
- Less boilerplate code

**Next Steps:**
- Experiment with different hyperparameters
- Try longer training (more epochs)
- Compare with Transformer architecture
- Use learning rate finder: `learn.lr_find()`